In [1]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")      # non-GUI backend, can't crash the kernel
import matplotlib.pyplot as plt
%matplotlib inline

full_df = pd.read_csv(r"C:\Users\sarav\projects\vowel-statistical-analysis\data\phoible.csv")

C:\Users\sarav\AppData\Local\Temp\ipykernel_18412\48517285.py:7: DtypeWarning: Columns (4,7,8,11) have mixed types. Specify dtype option on import or set low_memory=False.
  full_df = pd.read_csv(r"C:\Users\sarav\projects\vowel-statistical-analysis\data\phoible.csv")


In [2]:
total_rows = len(full_df)
total_invs = full_df["InventoryID"].nunique()
total_langs = full_df["Glottocode"].nunique()

In [3]:
print(f"Rows: {total_rows}")
print(f"Distinct inventories: {total_invs}")
print(f"Distinct languages: {total_langs}")

Rows: 105467
Distinct inventories: 3020
Distinct languages: 2184


In [4]:
full_df["ISO6393"].nunique()


2098

In [5]:
iso_col = "ISO6393" 
glotto_col = "Glottocode"

iso_counts = full_df.groupby(iso_col)[glotto_col].nunique()

multi_glotto_isos = iso_counts[iso_counts > 1].index[:3]

examples = (
    full_df[full_df[iso_col].isin(multi_glotto_isos)][[iso_col, glotto_col, 'LanguageName']]
    .drop_duplicates()
    .sort_values(by=iso_col)
)

print(examples)

      ISO6393 Glottocode        LanguageName
7650      aer   east2379            ARRERNTE
75556     aer   east2379   Arrernte, Central
97958     aer   mpar1238    Central Arrernte
98045     aer   east2379    Eastern Arrernte
97826     amx   east2380  Eastern Anmatyerre
97870     amx   west2442  Western Anmatyerre
67816     boa   bora1263                Bora
67857     boa   mira1254              Miraña


In [6]:
glotto_inv_counts = full_df.groupby("Glottocode")["InventoryID"].nunique()

multi_inv_glottocodes = glotto_inv_counts[glotto_inv_counts >= 2].index[:3]

examples = (
    full_df[full_df["Glottocode"].isin(multi_inv_glottocodes)][["Glottocode", "InventoryID", "LanguageName"]]
    .drop_duplicates()
    .sort_values(by="Glottocode")
)

print(examples)

      Glottocode  InventoryID LanguageName
21532   abid1235          649       abidji
55176   abid1235         1526       Abidji
8693    abip1241          235       ABIPON
69215   abip1241         1914       Abipon
88680   abkh1244         2468       Abkhaz
92639   abkh1244         2552       Abkhaz


In [7]:
max_ids = full_df.groupby("Glottocode")["InventoryID"].max()
single_inv_df = full_df[full_df["InventoryID"].isin(max_ids)].copy()

print("Original rows:", len(full_df))
print("Filtered rows:", len(single_inv_df))
print("Unique Glottocodes:", single_inv_df["Glottocode"].nunique())
print("Unique Inventories:", single_inv_df["InventoryID"].nunique())

Original rows: 105467
Filtered rows: 75370
Unique Glottocodes: 2184
Unique Inventories: 2184


In [8]:
multi = full_df.groupby("Glottocode")["InventoryID"].nunique().loc[lambda x: x > 1].index

for g in multi.to_series().sample(5, random_state=0):
    invs = {i: set(grp["Phoneme"]) for i, grp in full_df[full_df["Glottocode"] == g].groupby("InventoryID")}
    shared, union = set.intersection(*invs.values()), set.union(*invs.values())
    print(g, "sizes:", [len(s) for s in invs.values()], "shared:", len(shared), "union:", len(union))
    print("   differing:", list(union - shared)[:10])

seco1241 sizes: [18, 26] shared: 18 union: 26
   differing: ['ɨ̃', 'ũ', 'ã', 'n', 'ĩ', 'ẽ', 't̠ʃ', 'õ']
stan1293 sizes: [40, 39, 39, 44, 41, 45, 39, 44, 40] shared: 21 union: 95
   differing: ['oʊ', 'pʰ', 'æ', 'ɔ', 'a', 'o̞ː', 'aʊ', 'ɹ', 'ʊə', 'oː']
lako1247 sizes: [36, 36] shared: 26 union: 46
   differing: ['ɛ', 'z̪|z', 'tʰ', 'z', 'l̪|l', 't̪ʼ|tʼ', 't̪ʰ|tʰ', 'e̞', 'tʼ', 'n̪|n']
iris1253 sizes: [68, 69, 50, 49, 52] shared: 11 union: 139
   differing: ['æː', 'fˠ', 'ã', 'n̥ˠ', 't̪ʰ', 'd', 'ɸʲ', 'ɟʝ', 'pʲʰ', 'ɲ']
taga1270 sizes: [28, 23, 24, 26] shared: 10 union: 47
   differing: ['ɛ', 'ä', 'ʃ', 'a', 'd', 'ɲ', 'c', 'oː', 'ɪ', 'e']


In [9]:
inv_sizes = full_df.groupby(["Glottocode", "InventoryID"])["Phoneme"].count()
span = inv_sizes.groupby("Glottocode").agg(["min", "max", "count"])
multi = span[span["count"] > 1]
print("average size gap:", (multi["max"] - multi["min"]).mean())

average size gap: 7.497175141242938


In [10]:
sizes = full_df.groupby("InventoryID").agg(Source=("Source", "first"), size=("Phoneme", "count"))
print(sizes.groupby("Source")["size"].describe())

        count       mean        std   min   25%   50%    75%    max
Source                                                             
aa      203.0  39.724138   8.771734  22.0  34.0  38.0  44.00   82.0
ea      390.0  43.292308  14.319767  19.0  34.0  40.0  49.75  133.0
er      392.0  24.038265   4.836568  16.0  21.0  23.0  26.00   44.0
gm      460.0  41.917391  14.541718  18.0  33.0  39.0  47.00  161.0
ph      389.0  34.089974  11.452035  14.0  25.0  33.0  41.00   90.0
ra      100.0  42.610000   8.491106  21.0  36.0  42.0  48.00   62.0
saphon  355.0  25.495775   6.184714  11.0  21.0  25.0  29.00   51.0
spa     197.0  38.406091  12.957055  17.0  28.0  37.0  45.00   94.0
upsid   451.0  30.966741  11.554364  11.0  23.5  29.0  36.00  141.0
uz       83.0  44.686747  14.378955  21.0  34.5  41.0  55.50   74.0


In [11]:
#sizes.boxplot(column="size", by="Source", rot=45)
#plt.savefig("../results/size_by_source.pdf", dpi=150, bbox_inches="tight")

In [12]:
before = full_df.groupby("InventoryID")["Source"].first().value_counts(normalize=True)
after = single_inv_df.groupby("InventoryID")["Source"].first().value_counts(normalize=True)

comp = pd.DataFrame({"full": before, "filtered": after})
comp["change"] = comp["filtered"] - comp["full"]
print(comp.sort_values("change"))

            full  filtered    change
Source                              
spa     0.065232  0.009158 -0.056074
upsid   0.149338  0.108974 -0.040363
uz      0.027483  0.012821 -0.014663
ra      0.033113  0.029304 -0.003809
aa      0.067219  0.070055  0.002836
ph      0.128808  0.132784  0.003976
ea      0.129139  0.138736  0.009597
er      0.129801  0.155220  0.025418
saphon  0.117550  0.150641  0.033091
gm      0.152318  0.192308  0.039990


In [13]:
full_df.groupby("Source")["InventoryID"].agg(["min", "max", "median"]).sort_values("median")

,min,max,median
Source,,,
spa,1,197,105.0
upsid,198,648,422.0
aa,649,851,748.0
ph,852,1240,1047.0
gm,1241,1700,1458.0
ra,1701,1800,1750.0
saphon,1801,2155,1977.0
uz,2156,2238,2195.0
ea,2239,2628,2442.0


In [14]:
vowels = full_df[full_df["SegmentClass"] == "vowel"]

print(vowels["Marginal"].value_counts(dropna=False))
print()
print("proportions:")
print(vowels["Marginal"].value_counts(dropna=False, normalize=True).round(3))

Marginal
False    24135
NaN       6739
True       189
Name: count, dtype: int64

proportions:
Marginal
False    0.777
NaN      0.217
True     0.006
Name: proportion, dtype: float64


In [15]:
vowels = full_df[full_df["SegmentClass"] == "vowel"]

by_source = vowels.groupby("Source")["Marginal"].value_counts(dropna=False).unstack(fill_value=0)
print(by_source)

Marginal   NaN  False  True
Source                     
aa           0   2671     0
ea           0   5246    21
er           0   2025     0
gm           0   5198    32
ph           0   4017    45
ra        1319      0     0
saphon    3342      0     0
spa       2078      0     0
upsid        0   3765    68
uz           0   1213    23


In [16]:
vowel_df = single_inv_df[single_inv_df["SegmentClass"] == "vowel"].copy()

In [17]:
vowel_counts = vowel_df.groupby("Glottocode").size()

print(vowel_counts.describe())

count    2184.000000
mean       10.192766
std         6.243709
min         2.000000
25%         6.000000
50%         9.000000
75%        13.000000
max        50.000000
dtype: float64


In [18]:
#vowel_counts.plot.hist(bins=range(vowel_counts.min(), vowel_counts.max() + 2))
#plt.savefig("../results/vowel_dist.pdf", bbox_inches="tight")

In [19]:
print(single_inv_df["Glottocode"].nunique(), "->", vowel_df["Glottocode"].nunique())

# all languages have at least one vowel

2184 -> 2184


In [20]:
q1 = vowel_counts.quantile(0.25)
q3 = vowel_counts.quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

outliers = vowel_counts[(vowel_counts < lower) | (vowel_counts > upper)]
print(f"bounds: {lower} to {upper}")
print(f"{len(outliers)} outliers")
print(outliers.sort_values(ascending=False))

bounds: -4.5 to 23.5
99 outliers
Glottocode
elfd1234    50
scot1245    49
juho1239    46
para1301    43
dann1241    40
            ..
kuoo1238    24
mono1269    24
bamb1269    24
koon1245    24
kare1338    24
Length: 99, dtype: int64


In [21]:
top3 = vowel_df.groupby("Phoneme")["Glottocode"].nunique().sort_values(ascending=False).head(3)
print(top3)

Phoneme
i    2059
u    1966
a    1922
Name: Glottocode, dtype: int64


In [22]:
n = vowel_df["Glottocode"].nunique()
for v in ["i", "a", "u"]:
    langs = vowel_df[vowel_df["Phoneme"] == v]["Glottocode"].nunique()
    print(f"{v}: {langs/n:.1%}")

i: 94.3%
a: 88.0%
u: 90.0%


In [23]:
import unicodedata

def base(p):
    # keep only the core letter: drop combining marks and length/modifier symbols
    return "".join(c for c in unicodedata.normalize("NFD", p)
                    if unicodedata.category(c) != "Mn" and c not in "ːˑ̃")

vowel_df["base"] = vowel_df["Phoneme"].apply(base)

n = vowel_df["Glottocode"].nunique()
for v in ["i", "a", "u"]:
    langs = vowel_df[vowel_df["base"] == v]["Glottocode"].nunique()
    print(f"{v} (broad): {langs/n:.1%}")

i (broad): 97.2%
a (broad): 93.2%
u (broad): 92.9%


In [24]:
core = ["high", "low", "front", "back", "round"]
n = vowel_df["Glottocode"].nunique()

for v in ["i", "a", "u"]:
    ref = vowel_df[vowel_df["Phoneme"] == v][core].iloc[0]
    match = vowel_df[(vowel_df[core] == ref).all(axis=1) & (vowel_df["base"] == v)]
    strip = vowel_df[vowel_df["base"] == v]

    print(f"\n=== {v} === {match['Glottocode'].nunique()/n:.1%}")
    print("  counts as", v, ":", sorted(set(match['Phoneme']))[:12])
    print("  base-letter but different features (now excluded):",
          sorted(set(strip['Phoneme']) - set(match['Phoneme']))[:12])


=== i === 97.2%
  counts as i : ['i', 'iː', 'iˑ', 'ĩ', 'ĩː', 'ĭ', 'i̘', 'i̙', 'i̞', 'i̞ː', 'ĩ̞', 'i̠']
  base-letter but different features (now excluded): ['ï', 'i̜', 'i̽']

=== a === 92.6%
  counts as a : ['a', 'aː', 'aˑ', 'ã', 'ãː', 'ã̈', 'ã̈ː', 'ă', 'ä', 'äː', 'å', 'a̙']
  base-letter but different features (now excluded): ['a̟', 'a̟ː', 'ã̟', 'ã̟ː']

=== u === 92.8%
  counts as u : ['u', 'uː', 'uˑ', 'ũ', 'ũː', 'ŭ', 'u̘', 'u̙', 'u̞', 'u̞ː', 'u̞ˑ', 'u̠']
  base-letter but different features (now excluded): ['üː', 'u̜', 'u̟']


In [25]:
has = {v: set(vowel_df[vowel_df["base"] == v]["Glottocode"]) for v in ["i", "a", "u"]}
all_langs = set(vowel_df["Glottocode"])
violators = all_langs - (has["i"] & has["a"] & has["u"])

print(f"{len(violators)} violators out of {len(all_langs)} ({len(violators)/len(all_langs):.1%})\n")

for g in list(violators)[:5]:
    missing = [v for v in ["i", "a", "u"] if g not in has[v]]
    vowels = sorted(vowel_df[vowel_df["Glottocode"] == g]["Phoneme"])
    print(f"{g} | missing: {missing}")
    print(f"   vowels: {vowels}\n")

296 violators out of 2184 (13.6%)

gude1246 | missing: ['i', 'u']
   vowels: ['a', 'aː', 'ə', 'əː']

nort2795 | missing: ['a']
   vowels: ['e', 'i', 'o', 'u', 'ɑ']

choc1276 | missing: ['u']
   vowels: ['a', 'aː', 'i', 'iː', 'o', 'oː']

elfd1234 | missing: ['i', 'u']
   vowels: ['ai̯ː', 'auː', 'aː', 'ãɪ̯ː', 'ãː', 'i̯uo', 'i̯ũo', 'o̞', 'o̞ː', 'uo', 'uoː', 'ũo', 'ũoː', 'y', 'yœ', 'yœː', 'yː', 'ỹ', 'ỹœ', 'ỹœː', 'ỹː', 'æ', 'æː', 'œ', 'œː', 'œ̃', 'œ̃ː', 'ɐ', 'ɐ̃', 'ɔy̯ː', 'ɔ̞', 'ɔ̞ː', 'ɔ̞̃', 'ɔ̞̃ː', 'ɛ', 'ɛː', 'ɛ̃', 'ɛ̃ː', 'ɪ', 'ɪɛ', 'ɪɛː', 'ɪː', 'ɪ̃', 'ɪ̃ɛ', 'ɪ̃ɛː', 'ɪ̃ː', 'ʏ', 'ʏː', 'ʏ̃', 'ʏ̃ː']

kunj1245 | missing: ['i', 'u']
   vowels: ['a', 'o̞', 'ɛ', 'ɪ', 'ʊ']



In [36]:
equiv = {"i": {"i"}, "a": {"a"}, "u": {"u"}}
has = {v: set(vowel_df[vowel_df["base"].isin(syms)]["Glottocode"]) for v, syms in equiv.items()}

all_langs = set(vowel_df["Glottocode"])
violators = all_langs - (has["i"] & has["a"] & has["u"])

print(f"{len(violators)} violators out of {len(all_langs)} ({len(violators)/len(all_langs):.1%})\n")

for g in list(violators)[:5]:
    missing = [v for v in ["i", "a", "u"] if g not in has[v]]
    vowels = sorted(vowel_df[vowel_df["Glottocode"] == g]["Phoneme"])
    print(f"{g} | missing: {missing}")
    print(f"   vowels: {vowels}\n")

296 violators out of 2184 (13.6%)

gude1246 | missing: ['i', 'u']
   vowels: ['a', 'aː', 'ə', 'əː']

nort2795 | missing: ['a']
   vowels: ['e', 'i', 'o', 'u', 'ɑ']

choc1276 | missing: ['u']
   vowels: ['a', 'aː', 'i', 'iː', 'o', 'oː']

elfd1234 | missing: ['i', 'u']
   vowels: ['ai̯ː', 'auː', 'aː', 'ãɪ̯ː', 'ãː', 'i̯uo', 'i̯ũo', 'o̞', 'o̞ː', 'uo', 'uoː', 'ũo', 'ũoː', 'y', 'yœ', 'yœː', 'yː', 'ỹ', 'ỹœ', 'ỹœː', 'ỹː', 'æ', 'æː', 'œ', 'œː', 'œ̃', 'œ̃ː', 'ɐ', 'ɐ̃', 'ɔy̯ː', 'ɔ̞', 'ɔ̞ː', 'ɔ̞̃', 'ɔ̞̃ː', 'ɛ', 'ɛː', 'ɛ̃', 'ɛ̃ː', 'ɪ', 'ɪɛ', 'ɪɛː', 'ɪː', 'ɪ̃', 'ɪ̃ɛ', 'ɪ̃ɛː', 'ɪ̃ː', 'ʏ', 'ʏː', 'ʏ̃', 'ʏ̃ː']

kunj1245 | missing: ['i', 'u']
   vowels: ['a', 'o̞', 'ɛ', 'ɪ', 'ʊ']



In [33]:
candidates = {
    "i": ["ɪ", "ɨ", "y", "e"],
    "u": ["ʊ", "ɯ", "o"],
    "a": ["æ", "ä", "ɐ", "ɑ"],
}

for base_v, cands in candidates.items():
    base_langs = set(vowel_df[vowel_df["base"] == base_v]["Glottocode"])
    print(f"\n=== {base_v} (in {len(base_langs)} langs) ===")
    for c in cands:
        c_langs = set(vowel_df[vowel_df["base"] == c]["Glottocode"])
        both = base_langs & c_langs
        only_c = c_langs - base_langs
        print(f"  {c}: both={len(both)}  has {c} but not {base_v}={len(only_c)}")


=== i (in 2122 langs) ===
  ɪ: both=274  has ɪ but not i=41
  ɨ: both=382  has ɨ but not i=9
  y: both=120  has y but not i=4
  e: both=1622  has e but not i=25

=== u (in 2029 langs) ===
  ʊ: both=249  has ʊ but not u=49
  ɯ: both=112  has ɯ but not u=19
  o: both=1515  has o but not u=98

=== a (in 2035 langs) ===
  æ: both=115  has æ but not a=47
  ä: both=0  has ä but not a=0
  ɐ: both=34  has ɐ but not a=17
  ɑ: both=73  has ɑ but not a=117


In [28]:
n = vowel_df["Glottocode"].nunique()

for v in ["i", "a", "u"]:
    has_v = vowel_df[vowel_df["base"] == v]["Glottocode"].nunique()
    print(f"lacking {v}: {(n - has_v)/n:.1%}")

lacking i: 2.8%
lacking a: 6.8%
lacking u: 7.1%


In [29]:
equiv = {"i": {"i"}, "a": {"a", "ɑ"}, "u": {"u"}}
for v in ["i", "a", "u"]:
    has_v = vowel_df[vowel_df["base"].isin(equiv[v])]["Glottocode"].nunique()
    print(f"lacking {v}: {(n - has_v)/n:.1%}")

lacking i: 2.8%
lacking a: 1.5%
lacking u: 7.1%


In [40]:
for g in list(violators)[:15]:
    sub = vowel_df[vowel_df["Glottocode"] == g]
    print(f"\n{g} | source: {sub['Source'].iloc[0]} | {len(sub)} vowels")
    print(sub[["Phoneme", "Marginal"]].to_string(index=False))


gude1246 | source: gm | 4 vowels
Phoneme Marginal
      a    False
     aː    False
      ə    False
     əː    False

nort2795 | source: gm | 5 vowels
Phoneme Marginal
      ɑ    False
      e    False
      i    False
      o    False
      u    False

choc1276 | source: ph | 6 vowels
Phoneme Marginal
      a    False
     aː    False
      i    False
     iː    False
      o    False
     oː    False

elfd1234 | source: ea | 50 vowels
Phoneme Marginal
     aː    False
    ãː    False
      æ    False
     æː    False
   ai̯ː    False
  ãɪ̯ː    False
    auː    False
      ɐ    False
     ɐ̃    False
      ɛ    False
     ɛ̃    False
     ɛː    False
    ɛ̃ː    False
   i̯uo    False
  i̯ũo    False
      ɪ    False
     ɪ̃    False
     ɪː    False
    ɪ̃ː    False
     ɪɛ    False
    ɪ̃ɛ    False
    ɪɛː    False
   ɪ̃ɛː    False
     o̞    False
    o̞ː    False
      œ    False
     œ̃    False
     œː    False
    œ̃ː    False
     ɔ̞    False
    ɔ̞̃    False
    ɔ̞ː    Fa

In [41]:
# nasal / long from feature columns; front rounded from features (front, round, high/low span)
nasal_langs = set(vowel_df[vowel_df["nasal"] == "+"]["Glottocode"])
long_langs  = set(vowel_df[vowel_df["long"] == "+"]["Glottocode"])
frontrounded_langs = set(vowel_df[(vowel_df["front"] == "+") &
                                  (vowel_df["round"] == "+")]["Glottocode"])

print("nasal:", len(nasal_langs))
print("long:", len(long_langs))
print("front rounded:", len(frontrounded_langs))

nasal: 473
long: 821
front rounded: 173


In [42]:
for feat, langs in [("nasal", nasal_langs), ("long", long_langs)]:
    ok = exc = 0
    for g in langs:         # only languages that have the marked vowel
        grp = vowel_df[vowel_df["Glottocode"] == g]
        marked_bases = set(grp[grp[feat] == "+"]["base"])
        orals = set(grp[grp[feat] != "+"]["base"])
        if marked_bases <= orals: ok += 1
        else: exc += 1
    print(f"{feat}: {ok/(ok+exc):.1%} satisfy, {exc} exceptions")

# front rounded → i and u 
ok = exc = 0
for g in frontrounded_langs:
    bases = set(vowel_df[vowel_df["Glottocode"] == g]["base"])
    if {"i", "u"} <= bases: ok += 1
    else: exc += 1
print(f"front-rounded: {ok/(ok+exc):.1%} satisfy, {exc} exceptions")

nasal: 91.1% satisfy, 42 exceptions
long: 81.6% satisfy, 151 exceptions
front-rounded: 91.9% satisfy, 14 exceptions


In [44]:
for g in frontrounded_langs:
    bases = set(vowel_df[vowel_df["Glottocode"] == g]["base"])
    if not ({"i","u"} <= bases):
        missing = {"i","u"} - bases
        rounded = sorted(set(vowel_df[(vowel_df["Glottocode"] == g) &
                                      (vowel_df["front"] == "+") &
                                      (vowel_df["round"] == "+")]["Phoneme"]))
        print(f"{g} | missing: {missing} | front rounded: {rounded}")

elfd1234 | missing: {'u', 'i'} | front rounded: ['y', 'yœ', 'yœː', 'yː', 'ỹ', 'ỹœ', 'ỹœː', 'ỹː', 'œ', 'œː', 'œ̃', 'œ̃ː', 'ʏ', 'ʏː', 'ʏ̃', 'ʏ̃ː']
cent1973 | missing: {'u', 'i'} | front rounded: ['œː']
tibe1272 | missing: {'u'} | front rounded: ['yː', 'ỹː', 'œː', 'œ̃ː']
wari1268 | missing: {'u'} | front rounded: ['y', 'ø']
west2368 | missing: {'u'} | front rounded: ['yː']
vach1239 | missing: {'u'} | front rounded: ['yˑ', 'øˑ', 'ʏ']
lowa1242 | missing: {'u', 'i'} | front rounded: ['y', 'œ']
hopi1249 | missing: {'u'} | front rounded: ['ø']
yape1248 | missing: {'u'} | front rounded: ['œ', 'œː']
tzel1254 | missing: {'u', 'i'} | front rounded: ['y']
east2337 | missing: {'i'} | front rounded: ['y', 'yː']
orow1243 | missing: {'u'} | front rounded: ['ø', 'ʏ']
plau1238 | missing: {'u'} | front rounded: ['y']
uigh1240 | missing: {'u', 'i'} | front rounded: ['ø', 'œː', 'ʏ', 'ʏː']


In [ ]:
all_langs = set(vowel_df["Glottocode"])
A = frontrounded_langs                       # has front rounded
B = {g for g in all_langs                    # has both i and u
     if {"i","u"} <= set(vowel_df[vowel_df["Glottocode"]==g]["base"])}

tab = pd.crosstab(
    pd.Series([g in A for g in all_langs], name="has front-rounded"),
    pd.Series([g in B for g in all_langs], name="has i and u"))
print(tab)

has i and u        False  True 
has front-rounded              
False                154   1857
True                  14    159
